# 02 · Text Preprocessing

**Goal:** turn raw tweets into clean text that the sentiment models in notebook 03 can use.

Raw tweets contain a lot of noise that carries no sentiment: URLs (`http://t.co/abc123`), mentions (`@VirginAmerica`), hashtag symbols, HTML entities (`&amp;`), emoji, mixed case and repeated punctuation.

## Steps
1. Load the data from notebook 01
2. Inspect example tweets
3. Build the cleaning function step by step
4. Apply it to all tweets
5. Tokenize and remove stopwords
6. Quality check
7. Save the clean data for modeling

## 1. Setup

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
from nltk.corpus import stopwords

sys.path.insert(0, str(Path.cwd().parent))
from src.data import RAW_PATH, build_dashboard_data
from src.predict import NEGATION_WORDS, clean_text, to_tokens

pd.set_option("display.max_colwidth", 200)

## 2. Load data

The cleaned dataset from notebook 01: duplicates removed and the label artifacts in the text restored ("Cancelled Flightled" → "cancelled"). Removing duplicates before the train-test split also keeps the same tweet from landing in both sets.

In [2]:
df = build_dashboard_data(pd.read_csv(RAW_PATH))[["tweet_id", "airline", "sentiment", "negative_reason", "text"]]
print(f"{len(df):,} tweets")
df.head(3)

14,485 tweets


,tweet_id,airline,sentiment,negative_reason,text
0,567588278875213824,JetBlue,neutral,NaN,@JetBlue's new CEO seeks the right balance to please passengers and Wall ... - Greenfield Daily Reporter http://t.co/LM3opxkxch
1,567590027375702016,JetBlue,negative,Can't Tell,@JetBlue is REALLY getting on my nerves !! 😡😡 #nothappy
2,567591480085463040,United,negative,Late Flight,@united yes. We waited in line for almost an hour to do so. Some passengers just left not wanting to wait past 1am.


## 3. Example tweets

A random sample shows which patterns need cleaning.

In [3]:
# A random sample shows the kinds of noise to clean
sample_tweets = df['text'].sample(10, random_state=7).tolist()
for i, tweet in enumerate(sample_tweets, 1):
    print(f"{i}. {tweet}")
    print()

1. @SouthwestAir you could be a little more accommodating with extending my flight credit.Whats 10 extra days!#notahappytraveler

2. @SouthwestAir got mine!  Just in time for the wife and my 15th anniversary!

3. @united thanks

4. @USAirways what is the holdup? Should I be looking for another flight?

5. @united This is with regard to a flight from a couple weeks ago. I'm very frustrated with your policies & expressing that.

6. @SouthwestAir why do your MDW gates have so many fewer seats than your planes?

7. @SouthwestAir   Where the heck are the bags for Chicago 408. It has been an hour

8. @united no wheelchairs

9. @AmericanAir - answer the phones.  Trying to change a reservation and haven't been able to get a human to pick up in two days.

10. @united here is the ticket # 0162424965446 please refund my unnecessary upgrade fee



**Noise found in the sample:**
- URLs (`http://t.co/...`)
- Mentions (`@VirginAmerica`, `@united`)
- HTML entities and retweet markers (`&amp;`, `RT`) and extra spaces
- Mixed upper and lower case
- Repeated punctuation (`!!!`, `?????`)
- Emoji and symbols

## 4. Cleaning, step by step

Each step is applied to one example tweet so its effect is visible.

In [4]:
# One noisy example tweet
test_tweet = "@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... and you didn't even call http://t.co/abc123 #worstairline RT &amp; please respond :("
print(f"Original:\n{test_tweet}\n")

Original:
@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... and you didn't even call http://t.co/abc123 #worstairline RT &amp; please respond :(



In [5]:
# Step 1: Lowercase
step1 = test_tweet.lower()
print(f"After lowercase:\n{step1}\n")

After lowercase:
@virginamerica what a terrible flight!!! delayed again... and you didn't even call http://t.co/abc123 #worstairline rt &amp; please respond :(



In [6]:
# Step 2: Remove URLs
step2 = re.sub(r'http\S+|www\S+|https\S+', '', step1)
print(f"After URL removal:\n{step2}\n")

After URL removal:
@virginamerica what a terrible flight!!! delayed again... and you didn't even call  #worstairline rt &amp; please respond :(



In [7]:
# Step 3: Remove mentions (@username)
step3 = re.sub(r'@\w+', '', step2)
print(f"After mention removal:\n{step3}\n")

After mention removal:
 what a terrible flight!!! delayed again... and you didn't even call  #worstairline rt &amp; please respond :(



In [8]:
# Step 4: Remove hashtag symbols (keep the word)
# '#worstairline' becomes 'worstairline': the word carries sentiment
step4 = re.sub(r'#', '', step3)
print(f"After hashtag handling:\n{step4}\n")

After hashtag handling:
 what a terrible flight!!! delayed again... and you didn't even call  worstairline rt &amp; please respond :(



In [9]:
# Step 5: Decode HTML entities and expand contractions
# "didn't" -> "did not": otherwise the apostrophe is stripped, "didn" becomes a
# separate token, and the stopword list removes it, so the negation is lost.
step5 = step4.replace("&amp;", "&").replace("&", " and ")
step5 = re.sub(r"\bcan't\b", "can not", step5)
step5 = re.sub(r"\bwon't\b", "will not", step5)
step5 = re.sub(r"n't\b", " not", step5)
print(f"After HTML entities and contractions:\n{step5}\n")

After HTML entities and contractions:
 what a terrible flight!!! delayed again... and you did not even call  worstairline rt  and  please respond :(



In [10]:
# Step 6: Remove special chars, keep only letters & spaces
step6 = re.sub(r'[^a-z\s]', ' ', step5)
print(f"After special char removal:\n{step6}\n")

After special char removal:
 what a terrible flight    delayed again    and you did not even call  worstairline rt  and  please respond   



In [11]:
# Step 7: Remove extra whitespace
step7 = re.sub(r'\s+', ' ', step6).strip()
print(f"After whitespace cleanup:\n{step7}\n")

After whitespace cleanup:
what a terrible flight delayed again and you did not even call worstairline rt and please respond



**Before:** `@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... and you didn't even call http://t.co/abc123 #worstairline RT &amp; please respond :(`

**After:** `what a terrible flight delayed again and you did not even call worstairline rt and please respond`

The words that carry sentiment (*terrible*, *delayed*, *did not*, *worstairline*) survive; the noise is gone.

## 5. Final cleaning function

The steps above are combined in `clean_text` in `src/predict.py`. The notebook imports it rather than redefining it, so the training data and the dashboard's model tab clean text in exactly the same way.

In [12]:
import inspect

print(inspect.getsource(clean_text))
assert clean_text(test_tweet) == step7
print(clean_text(test_tweet))

def clean_text(text: str) -> str:
    """Clean a tweet for the models (used by notebook 02 and the dashboard).

    Contractions are expanded first ("didn't" -> "did not"). Otherwise the
    apostrophe is stripped, "didn" ends up as a separate token, and NLTK's
    stopword list removes it, so the negation would be lost.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower().replace("’", "'")
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = text.replace("&amp;", "&").replace("&lt;", "").replace("&gt;", "").replace("&quot;", "")
    text = text.replace("&", " and ")
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

what a terrible flight delayed again and you did not even call worstairline rt an

## 6. Apply cleaning to all tweets

In [13]:
df["text_clean"] = df["text"].map(clean_text)

for _, row in df[["text", "text_clean"]].sample(5, random_state=42).iterrows():
    print(f"Original: {row['text']}")
    print(f"Cleaned:  {row['text_clean']}")
    print("-" * 100)

Original: @united why are your agents working so slowly to rebook people who are on #UA1481. We have all wasted an entire day at STT.
Cleaned:  why are your agents working so slowly to rebook people who are on ua we have all wasted an entire day at stt
----------------------------------------------------------------------------------------------------
Original: @USAirways @AmericanAir 9 empty seats open in 1st on AA1061. No upgrades for a U.S. Gold member? Would love consistent policies.
Cleaned:  empty seats open in st on aa no upgrades for a u s gold member would love consistent policies
----------------------------------------------------------------------------------------------------
Original: @JetBlue the big board says we aren't leaving until after 8:30 😩
Cleaned:  the big board says we are not leaving until after
----------------------------------------------------------------------------------------------------
Original: @JetBlue thanks so much for help Us, u r amazing!
Cleane

## 7. Tokenization and stopword removal

- **Tokenization** splits text into words.
- **Stopword removal** drops very common words (*the, is, a*) that carry no sentiment, using NLTK's English list **minus negation words** such as *not* and *no*, because they reverse sentiment.

Two versions of the text are kept:
- `text_clean` (full sentences) for **VADER** and **DistilBERT**, which rely on sentence context.
- `text_tokens` (without stopwords) for **Naive Bayes** and **Logistic Regression** on TF-IDF features. It is produced by `to_tokens` in `src/predict.py`, the same function the dashboard uses.

In [14]:
english = set(stopwords.words("english"))
print(f"NLTK English stopwords: {len(english)}")
print(f"Kept because they carry negation: {sorted(NEGATION_WORDS & english)}")
print()
print(to_tokens("The crew wasn't helpful and didn't answer"))

NLTK English stopwords: 198
Kept because they carry negation: ['no', 'nor', 'not']

crew not helpful not answer


In [15]:
df["text_tokens"] = df["text"].map(to_tokens)

for _, row in df[["text_clean", "text_tokens"]].sample(5, random_state=42).iterrows():
    print(f"Before: {row['text_clean']}")
    print(f"After:  {row['text_tokens']}")
    print("-" * 100)

Before: why are your agents working so slowly to rebook people who are on ua we have all wasted an entire day at stt
After:  agents working slowly rebook people ua wasted entire day stt
----------------------------------------------------------------------------------------------------
Before: empty seats open in st on aa no upgrades for a u s gold member would love consistent policies
After:  empty seats open st aa no upgrades gold member would love consistent policies
----------------------------------------------------------------------------------------------------
Before: the big board says we are not leaving until after
After:  big board says not leaving
----------------------------------------------------------------------------------------------------
Before: thanks so much for help us u r amazing
After:  thanks much help us amazing
----------------------------------------------------------------------------------------------------
Before: still crickets from customer service
A

## 8. Quality check

In [16]:
print(f"Tweets with empty text_clean:  {(df['text_clean'].str.len() == 0).sum()}")
print(f"Tweets with empty text_tokens: {(df['text_tokens'].str.len() == 0).sum()}")
print()
print(f"Average length of text_clean:  {df['text_clean'].str.len().mean():.1f} characters")
print(f"Average length of text_tokens: {df['text_tokens'].str.len().mean():.1f} characters")
print(f"Average number of tokens:      {df['text_tokens'].str.split().str.len().mean():.1f}")

Tweets with empty text_clean:  0
Tweets with empty text_tokens: 21

Average length of text_clean:  83.8 characters
Average length of text_tokens: 56.2 characters
Average number of tokens:      8.8


In [17]:
# Empty tweets carry no signal for the models, so they are dropped
before = len(df)
df = df[df["text_tokens"].str.len() > 0].reset_index(drop=True)
print(f"Dropped {before - len(df)} empty tweets, {len(df):,} left")

Dropped 21 empty tweets, 14,464 left


## 9. Save clean data

Saved to `data/processed/tweets_clean.csv` for notebook 03.

In [18]:
df.to_csv("../data/processed/tweets_clean.csv", index=False)
print(f"Saved data/processed/tweets_clean.csv: {df.shape[0]:,} rows, columns {list(df.columns)}")

Saved data/processed/tweets_clean.csv: 14,464 rows, columns ['tweet_id', 'airline', 'sentiment', 'negative_reason', 'text', 'text_clean', 'text_tokens']


## Summary

**Cleaning applied:**
- Lowercasing
- Removal of URLs, mentions, hashtag symbols and HTML entities
- Expansion of contractions (*didn't* → *did not*)
- Removal of special characters and extra whitespace
- Tokenization and stopword removal, **keeping negation words** such as *not* and *no*, because they reverse sentiment

Expanding contractions matters: **13.4% of tweets (1,938) contain one such as *didn't* or *can't*, and 84% of those are negative.** Without the expansion the apostrophe is stripped, NLTK's stopword list removes the leftover *didn*, and the model never sees the negation.

**Output:** `data/processed/tweets_clean.csv` with 14,464 tweets (21 that were empty after stopword removal were dropped).

**Next, notebook 03:** four sentiment models are trained and compared on the same test set:
1. **VADER** (rule-based) on `text_clean`
2. **Naive Bayes** (classical ML) on `text_tokens`
3. **Logistic Regression** (classical ML) on `text_tokens`
4. **DistilBERT** (pre-trained deep learning) on `text_clean`